# Fine-tuning Qwen3-8B with QLoRA on Amazon EKS

This notebook provides a comprehensive guide to fine-tuning the **Qwen3-8B** large language model using **QLoRA** (Quantized Low-Rank Adaptation) on **Amazon EKS** (Elastic Kubernetes Service).

## What You'll Learn

1. **Understanding QLoRA**: How to efficiently fine-tune large models with limited GPU memory
2. **Amazon EKS Basics**: How to set up a Kubernetes cluster with GPU support
3. **Medical Reasoning Dataset**: Preparing data for instruction tuning
4. **Training Pipeline**: End-to-end training with monitoring
5. **Model Deployment**: Using the fine-tuned model for inference

## Prerequisites

- AWS Account with appropriate permissions
- Basic familiarity with Python and command line
- (Optional) GPU machine for local testing

---

# Part 1: Understanding QLoRA

## What is QLoRA?

**QLoRA** (Quantized Low-Rank Adaptation) is a technique that enables fine-tuning of large language models on consumer-grade GPUs by combining:

1. **4-bit Quantization**: Compresses model weights from 32/16 bits to 4 bits
2. **LoRA (Low-Rank Adaptation)**: Adds small trainable adapters instead of updating all weights
3. **Paged Optimizers**: Efficiently manages GPU memory during training

### Memory Comparison

| Model Size | Full Fine-tuning | QLoRA |
|------------|-----------------|-------|
| 7B params  | ~28 GB          | ~6 GB |
| 8B params  | ~32 GB          | ~8 GB |
| 13B params | ~52 GB          | ~12 GB|
| 70B params | ~280 GB         | ~48 GB|

### How LoRA Works

Instead of updating all weights `W`, LoRA learns a low-rank decomposition:

```
W' = W + BA
```

Where:
- `W` is the frozen original weight (frozen, quantized to 4-bit)
- `B` is a trainable matrix of shape (d, r)
- `A` is a trainable matrix of shape (r, k)
- `r` is the rank (typically 8-64, much smaller than d, k)

## Key QLoRA Parameters

### Quantization Parameters

- **`load_in_4bit`**: Enable 4-bit quantization
- **`bnb_4bit_quant_type`**: "nf4" (NormalFloat4) is optimal for LLM weights
- **`bnb_4bit_use_double_quant`**: Nested quantization for extra memory savings
- **`bnb_4bit_compute_dtype`**: Data type for computations (bfloat16 recommended)

### LoRA Parameters

- **`r` (rank)**: Higher values = more capacity, more memory. Typical: 8-64
- **`lora_alpha`**: Scaling factor, typically set to 2x rank
- **`target_modules`**: Which layers to apply LoRA to

In [ ]:
# Example QLoRA Configuration

# This is what the configuration looks like in code:
qlora_config = {
    # Quantization settings
    "load_in_4bit": True,
    "bnb_4bit_quant_type": "nf4",         # NormalFloat4 - optimal for LLMs
    "bnb_4bit_use_double_quant": True,    # Saves ~0.4 bits per parameter
    "bnb_4bit_compute_dtype": "bfloat16", # Computation precision
    
    # LoRA settings
    "lora_r": 16,                         # Rank of adaptation matrices
    "lora_alpha": 32,                     # Scaling factor
    "lora_dropout": 0.05,                 # Regularization
    "target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers
        "gate_proj", "up_proj", "down_proj"       # MLP layers
    ]
}

print("QLoRA Configuration:")
for key, value in qlora_config.items():
    print(f"  {key}: {value}")

---

# Part 2: Understanding Amazon EKS

## What is Amazon EKS?

**Amazon Elastic Kubernetes Service (EKS)** is a managed Kubernetes service that makes it easy to run Kubernetes on AWS without needing to maintain your own Kubernetes control plane.

### Why Use EKS for ML Training?

1. **Scalability**: Easily scale GPU nodes up/down based on demand
2. **Cost Efficiency**: Pay only for resources when training
3. **Reproducibility**: Containerized workloads ensure consistent environments
4. **Fault Tolerance**: Kubernetes handles failures and restarts
5. **Resource Management**: Efficient scheduling of GPU workloads

### EKS Architecture for ML

```
┌─────────────────────────────────────────────────────┐
│                    EKS Cluster                       │
│  ┌─────────────────┐  ┌─────────────────────────┐  │
│  │  System Nodes   │  │      GPU Nodes           │  │
│  │  (m5.large)     │  │   (g5.12xlarge)          │  │
│  │                 │  │                          │  │
│  │  - CoreDNS      │  │  ┌──────────────────┐   │  │
│  │  - VPC-CNI      │  │  │ Training Pod     │   │  │
│  │  - Monitoring   │  │  │ - GPU: 1         │   │  │
│  │                 │  │  │ - Memory: 24GBx4 │   │  │
│  │                 │  │  │ - Storage: PVC   │   │  │
│  │                 │  │  └──────────────────┘   │  │
│  └─────────────────┘  └─────────────────────────┘  │
│                                                     │
│  ┌─────────────────────────────────────────────┐   │
│  │              EBS Storage (PVC)               │   │
│  │  - Model checkpoints                         │   │
│  │  - Hugging Face cache                        │   │
│  │  - Training outputs                          │   │
│  └─────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────┘
```

---

# Part 3: The Medical Reasoning Dataset

We use the **medical-o1-reasoning-SFT** dataset from FreedomIntelligence, which contains:

- **19,700 samples** (English subset)
- Medical questions with detailed reasoning chains
- Based on Deepseek-R1 distillation

## Dataset Structure

Each sample has three fields:

1. **Question**: The medical question or clinical case
2. **Complex_CoT**: Chain-of-thought reasoning process
3. **Response**: The final answer or diagnosis

In [ ]:
# Install required packages (uncomment to run)
# !pip install datasets transformers

In [ ]:
# Load and explore the dataset
from datasets import load_dataset

# Load the English subset
dataset = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    name="en",
    split="train"
)

print(f"Dataset size: {len(dataset)} samples")
print(f"\nColumns: {dataset.column_names}")

In [ ]:
# View a sample
sample = dataset[0]

print("=" * 60)
print("QUESTION:")
print("=" * 60)
print(sample["Question"][:500] + "..." if len(sample["Question"]) > 500 else sample["Question"])

print("\n" + "=" * 60)
print("REASONING (Chain-of-Thought):")
print("=" * 60)
print(sample["Complex_CoT"][:800] + "..." if len(sample["Complex_CoT"]) > 800 else sample["Complex_CoT"])

print("\n" + "=" * 60)
print("RESPONSE:")
print("=" * 60)
print(sample["Response"][:500] + "..." if len(sample["Response"]) > 500 else sample["Response"])

## Data Formatting for Training

We format the data into a chat template that Qwen3 understands:

In [ ]:
# Chat template for Qwen3
CHAT_TEMPLATE = """<|im_start|>system
You are a medical AI assistant trained to provide detailed reasoning for medical questions.
Think through the problem step-by-step before providing your final answer.<|im_end|>
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
<think>
{reasoning}
</think>

{response}<|im_end|>"""

# Format a sample
formatted = CHAT_TEMPLATE.format(
    question=sample["Question"][:200],
    reasoning=sample["Complex_CoT"][:300],
    response=sample["Response"][:200]
)

print("Formatted sample (truncated):")
print(formatted)

---

# Part 4: Setting Up the Training Environment

## Option A: Local Training (GPU Required)

If you have a local GPU with 24GB+ VRAM:

In [ ]:
# Check GPU availability
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {gpu_mem:.1f} GB")
    
    if gpu_mem >= 20:
        print("\n✓ Sufficient VRAM for QLoRA training!")
    else:
        print("\n⚠ Limited VRAM - consider using smaller batch size or EKS")
else:
    print("No GPU detected. Use EKS for training.")

## Option B: EKS Training (Recommended for Production)

### Step 1: Install Required Tools

```bash
# Install AWS CLI
curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "awscliv2.zip"
unzip awscliv2.zip && sudo ./aws/install

# Install eksctl
curl --silent --location "https://github.com/eksctl-io/eksctl/releases/latest/download/eksctl_$(uname -s)_amd64.tar.gz" | tar xz -C /tmp
sudo mv /tmp/eksctl /usr/local/bin

# Install kubectl
curl -LO "https://dl.k8s.io/release/$(curl -L -s https://dl.k8s.io/release/stable.txt)/bin/linux/amd64/kubectl"
sudo install -o root -g root -m 0755 kubectl /usr/local/bin/kubectl
```

### Step 2: Configure AWS Credentials

```bash
aws configure
# Enter your AWS Access Key ID, Secret Access Key, and Region (us-west-2)
```

### Step 3: Create EKS Cluster

```bash
# From the project root directory
cd qwen3-qlora-eks-tutorial

# Run the setup script (takes 15-20 minutes)
./eks/setup-cluster.sh
```

This script will:
1. Create the EKS cluster with GPU nodes
2. Install NVIDIA device plugin
3. Create ECR repository
4. Set up storage classes

### Step 4: Build and Push Docker Image

```bash
# Build and push the training image
./scripts/build_and_push.sh
```

### Step 5: Deploy Training Job

```bash
# Deploy the training job
./scripts/deploy_training.sh

# Monitor training logs
kubectl logs -f job/qwen3-qlora-training -n ml-training
```

---

# Part 5: Training Code Walkthrough

Let's walk through the key parts of the training code:

In [ ]:
# Step 1: Load model with 4-bit quantization

# Note: This requires a GPU and will download the model (~16GB)
# Uncomment to run

'''
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Create quantization config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-8B",
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3-8B",
    trust_remote_code=True,
)
'''

print("Model loading code ready (uncomment to run with GPU)")

In [ ]:
# Step 2: Add LoRA adapters

'''
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Create LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Add LoRA adapters
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()
# Output: trainable params: 41,943,040 || all params: 8,251,297,792 || trainable%: 0.508
'''

print("LoRA adapter code ready (uncomment to run with GPU)")

In [ ]:
# Step 3: Training configuration

training_config = {
    "output_dir": "./outputs",
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,  # Effective batch size = 16
    "num_train_epochs": 3,
    "learning_rate": 2e-4,
    "warmup_ratio": 0.03,
    "lr_scheduler_type": "cosine",
    "optim": "paged_adamw_8bit",  # Memory-efficient optimizer
    "bf16": True,
    "gradient_checkpointing": True,
    "max_grad_norm": 0.3,
}

print("Training Configuration:")
for key, value in training_config.items():
    print(f"  {key}: {value}")

---

# Part 6: Monitoring Training

## Using TensorBoard

In [ ]:
# Start TensorBoard (run this in a terminal or uncomment below)
# !tensorboard --logdir ./outputs/logs

# Or use TensorBoard inline in Jupyter
# %load_ext tensorboard
# %tensorboard --logdir ./outputs/logs

print("TensorBoard commands:")
print("  Terminal: tensorboard --logdir ./outputs/logs")
print("  Then open: http://localhost:6006")

## Using kubectl for EKS

```bash
# Watch training logs
kubectl logs -f job/qwen3-qlora-training -n ml-training

# Check GPU utilization
kubectl exec -it $(kubectl get pods -n ml-training -l component=training -o jsonpath='{.items[0].metadata.name}') -n ml-training -- nvidia-smi

# Check pod status
kubectl get pods -n ml-training -w
```

---

# Part 7: Cleanup

**Important**: Delete resources when done to avoid charges!

```bash
# Delete all resources
./scripts/cleanup.sh

# Or manually:
eksctl delete cluster --name qwen3-qlora-cluster --region us-west-2
```

---

# Summary

In this tutorial, you learned:

1. **QLoRA Fundamentals**: How 4-bit quantization + LoRA enables efficient fine-tuning
2. **EKS Setup**: Creating a GPU-enabled Kubernetes cluster
3. **Data Preparation**: Formatting medical reasoning data for instruction tuning
4. **Training Pipeline**: End-to-end training with monitoring
5. **Inference**: Using the fine-tuned model for medical question answering

## Next Steps

- Experiment with different LoRA ranks (8, 32, 64)
- Try different learning rates (1e-4 to 5e-4)
- Fine-tune on your own domain-specific dataset
- Deploy the model for production inference

## Resources

- [QLoRA Paper](https://arxiv.org/abs/2305.14314)
- [Qwen3 Model Card](https://huggingface.co/Qwen/Qwen3-8B)
- [PEFT Documentation](https://huggingface.co/docs/peft)
- [Amazon EKS Documentation](https://docs.aws.amazon.com/eks/)
- [Medical Reasoning Dataset](https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT)